In [64]:
# Cell 1: Installation
!pip install sympy gradio matplotlib numpy -q

In [65]:
# Cell 2: Imports
import sympy as sp
import gradio as gr
import re
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io

# Define symbols
x, y, z, t = sp.symbols('x y z t')

# Set a nice style for our plots
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries imported and plotting style set!")

✅ Libraries imported and plotting style set!


In [66]:
# Cell 3: Core Logic with Step-by-Step Explanation

import random
import re

# --- 1. State Management ---
state = {
    "score": 0,
    "streak": 0,
    "active_q": None,
    "active_a": None
}

# --- 2. Helper Functions ---
def preprocess_input(user_input):
    expr = user_input.strip()
    expr = expr.replace('^', '**')
    expr = re.sub(r'(\d)([a-zA-Z])', r'\1*\2', expr)
    if '=' in expr:
        parts = expr.split('=')
        if len(parts) == 2:
            lhs = parts[0].strip()
            rhs = parts[1].strip()
            expr = f"({lhs}) - ({rhs})"
    return expr

# --- 3. The Main Solver (Detailed Steps) ---
def math_tutor_engine(user_problem):
    if not user_problem:
        return "No input."
    try:
        clean_expr = preprocess_input(user_problem)
        problem_type = "simplification"
        steps = []
        result = None

        # Detect Intent
        if "derive" in user_problem.lower() or "differentiate" in user_problem.lower():
            problem_type = "derivative"
        elif "integrate" in user_problem.lower():
            problem_type = "integral"
        elif "=" in user_problem:
            problem_type = "equation"

        # --- Process ---

        # Case A: Derivative
        if problem_type == "derivative":
            expr_str = clean_expr.replace("derive", "").replace("differentiate", "").strip()
            expr = sp.sympify(expr_str)
            result = sp.diff(expr, x)

            steps.append(f"1️⃣ **Input:** `{user_problem}`")
            steps.append(f"2️⃣ **Operation:** Differentiation (finding rate of change)")
            steps.append(f"3️⃣ **Applying Rules:** Applied Power Rule and Chain Rule on `{expr}`.")
            steps.append(f"4️⃣ **Result:** `{result}`")

        # Case B: Integral
        elif problem_type == "integral":
            expr_str = clean_expr.replace("integrate", "").strip()
            expr = sp.sympify(expr_str)
            result = sp.integrate(expr, x)

            steps.append(f"1️⃣ **Input:** `{user_problem}`")
            steps.append(f"2️⃣ **Operation:** Integration (finding area under curve)")
            steps.append(f"3️⃣ **Applying Rules:** Integrated `{expr}` with respect to `x`.")
            steps.append(f"4️⃣ **Result:** `{result}` + C (constant)")

        # Case C: Equation (Solver)
        elif problem_type == "equation":
            # Parse equation
            expr = sp.sympify(clean_expr)
            result = sp.solve(expr, x)

            steps.append(f"1️⃣ **Equation:** `{user_problem}`")
            steps.append(f"2️⃣ **Goal:** Find the value of `x`.")
            steps.append(f"3️⃣ **Method:** Moved terms to one side to isolate `x`.")
            steps.append(f"4️⃣ **Solution Found:** `{result}`")

        # Case D: Simplification / Arithmetic
        else:
            expr = sp.sympify(clean_expr)
            result = sp.simplify(expr)

            steps.append(f"1️⃣ **Input:** `{user_problem}`")
            steps.append(f"2️⃣ **Operation:** Simplification / Calculation")
            steps.append(f"3️⃣ **Process:** Evaluated the expression using standard order of operations.")
            steps.append(f"4️⃣ **Result:** `{result}`")

        # Format Final Output
        steps_text = "\n".join(steps)
        return f"✅ **Final Answer:** `{result}`\n\n---\n**📖 Explanation:**\n{steps_text}"

    except Exception as e:
        return f"❌ **Error:** {str(e)}"

# --- 4. Gamification Logic ---
def generate_challenge():
    a = random.randint(1, 10)
    b = random.randint(1, 10)
    if random.random() > 0.5:
        problem = f"{a}*x + {b} = 0"
        solution = sp.solve(sp.sympify(f"{a}*x + {b}"), x)
    else:
        problem = f"x**2 - {a} = 0"
        solution = sp.solve(sp.sympify(f"x**2 - {a}"), x)
    return problem, solution

def ask_challenge():
    problem, solution = generate_challenge()
    state["active_q"] = problem
    state["active_a"] = str(solution)
    return f"🎯 **Challenge:**\n\n{problem}\n\nSolve for `x`!"

def check_game_answer(user_text):
    user_text = str(user_text).strip()
    correct_answer = state["active_a"].strip()

    # Check answer
    if user_text in correct_answer or correct_answer in user_text:
        state["score"] += 10
        state["streak"] += 1
        return f"✅ **CORRECT!**\n\n🏆 **Score:** {state['score']}\n🔥 **Streak:** {state['streak']}"
    else:
        # If wrong, show the correct solution with steps
        explanation = math_tutor_engine(state["active_q"])
        state["streak"] = 0
        return f"❌ **Wrong!**\n\nThe correct answer was: {correct_answer}\n\n**See Solution Below:**\n{explanation}"

In [ ]:
# Cell 4: The Interface
import gradio as gr

def interact(message, history):
    # 1. Check if we are in Game Mode (waiting for answer)
    if state["active_q"] is not None:
        response = check_game_answer(message)
        state["active_q"] = None # Reset game mode after attempt
    else:
        # 2. Normal Math Mode
        response = math_tutor_engine(message)

    history.append((message, response))
    return history, f"🏆 Score: {state['score']}", f"🔥 Streak: {state['streak']}"

def switch_theme(theme_name):
    if "Midnight" in theme_name:
        return gr.themes.Soft(primary_hue="slate", neutral_hue="slate").set(body_background_fill="#0f172a")
    elif "Cyberpunk" in theme_name:
        return gr.themes.Soft(primary_hue="pink", secondary_hue="purple").set(body_background_fill="#1e1b4b")
    else:
        return gr.themes.Soft()

with gr.Blocks(theme=gr.themes.Soft()) as app:
    with gr.Row():
        # Sidebar
        with gr.Column(scale=1, min_width=250):
            gr.Markdown("### ⚙️ Settings")
            theme_drop = gr.Dropdown(["Ocean (Light)", "Midnight (Dark)", "Cyberpunk"], label="Change Theme", value="Ocean (Light)")
            gr.Markdown("---")
            gr.Markdown("### 🎮 Game Zone")
            game_btn = gr.Button("⚡ Challenge Me", variant="primary")
            score_display = gr.Markdown("🏆 Score: 0")
            streak_display = gr.Markdown("🔥 Streak: 0")

        # Main Chat
        with gr.Column(scale=4):
            gr.Markdown("<h1 style='text-align: center;'>🧮 Math Genius AI</h1>")
            chatbot = gr.Chatbot(height=500, show_label=False)
            with gr.Row():
                txt = gr.Textbox(show_label=False, placeholder="Type answer or math...", scale=9)
                btn = gr.Button("Send", variant="primary", scale=1)

    # Connections
    btn.click(interact, [txt, chatbot], [chatbot, score_display, streak_display])
    txt.submit(interact, [txt, chatbot], [chatbot, score_display, streak_display])

    # Connect your Challenge Logic here
    game_btn.click(ask_challenge, outputs=txt) # Puts question into input box

    theme_drop.change(switch_theme, inputs=[theme_drop], outputs=[app])

app.launch(share=True, debug=True)

/tmp/ipykernel_161/4147569862.py:24: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:
/tmp/ipykernel_161/4147569862.py:39: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=500, show_label=False)
/tmp/ipykernel_161/4147569862.py:39: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=500, show_label=False)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://36a6a376485cb41aa9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
